# ДЗ 4. Операции, axis и broadcasting на данных «Дала Маркет»
Срок: **до понедельника 16.11, 19:00**.
Сдача: папка `hw04` репозитория `dsa-homework` (`hw04.ipynb`): `git add hw04` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число + одна фраза для менеджера**. Циклы `for` по заказам не используем. Небольшой цикл по городам или категориям при сборке таблицы допустим.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import numpy as np
# 1
a = np.array([2, 4, 6]); print(a / 2, a // 4)
# 2
m = np.ones((3, 4)); print(m.sum(axis=1))
# 3
print(np.arange(3) + np.arange(3).reshape(3, 1))
# 4
x = np.array([1, 4, 9]); print(np.sqrt(x).sum())
# 5
print((np.arange(1, 5) ** 2).mean())
```

**Мои ответы (сначала предсказание, затем проверка):**

1. **`[1. 2. 3.] [0 1 1]`**. Деление `/` даёт числа с плавающей точкой, а `//` делит на 4 с округлением вниз.
2. **`[4. 4. 4.]`**. У матрицы 3 строки по четыре единицы; `axis=1` суммирует внутри каждой строки.
3. **`[[0 1 2], [1 2 3], [2 3 4]]`**. Массивы форм `(3,)` и `(3, 1)` складываются через broadcasting до формы `(3, 3)`.
4. **`6.0`**. Квадратные корни из 1, 4, 9 — это 1, 2, 3; их сумма равна 6.
5. **`7.5`**. Среднее квадратов 1, 2, 3, 4: `(1 + 4 + 9 + 16) / 4 = 7.5`.

In [6]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [7]:
import csv
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

amount = np.array([float(r["amount"].replace(" ", "")) for r in rows])
paid   = np.array([r["status"] == "оплачен" for r in rows])
days   = np.array([float(r["delivery_days"]) if r["delivery_days"] else np.nan for r in rows])
city   = np.array([CITY.get(r["city"].strip().lower(), r["city"].strip()) for r in rows])
cust   = np.array([r["customer_id"] for r in rows])
month  = np.array([r["order_date"][:7] for r in rows])      # "2026-09" или "2026-10"
cat    = np.array([r["category"] for r in rows])
oid    = np.array([r["order_id"] for r in rows])

print(amount.shape, np.unique(month))

(5463,) ['2026-09' '2026-10']


In [8]:
# Чистые данные: без дублей (оставляем первое вхождение order_id), без TEST, только оплаченные
_, first = np.unique(oid, return_index=True)
keep = np.zeros(len(oid), dtype=bool)
keep[first] = True
clean = keep & (cust != "TEST") & paid
print("строк всего:", len(oid), "| в чистых данных:", clean.sum())

строк всего: 5463 | в чистых данных: 4739


## [База] 1. Выручка по чистым данным
Сколько строк осталось в `clean`? Выручка за сентябрь и октябрь и изменение в %. Совпадает ли с ДЗ 2?

In [9]:
# 1. Выручка только по чистым оплаченным заказам
months = np.array(["2026-09", "2026-10"])
monthly_revenue = np.array([amount[clean & (month == m)].sum() for m in months])
monthly_change_pct = (monthly_revenue[1] / monthly_revenue[0] - 1) * 100
print(f"Чистых заказов: {clean.sum():,}")
print(f"Сентябрь: {monthly_revenue[0]:,.0f} ₸")
print(f"Октябрь:  {monthly_revenue[1]:,.0f} ₸")
print(f"Изменение: {monthly_change_pct:.2f}%")
# Проверяем числа обычной печатью
print("Проверка: должно быть 4739 и [35243410, 34419520]:")
print(clean.sum(), monthly_revenue)
print("Для менеджера: после удаления дублей, TEST и отмен выручка упала примерно на 2,3%, как в ДЗ 2.")

Чистых заказов: 4,739
Сентябрь: 35,243,410 ₸
Октябрь:  34,419,520 ₸
Изменение: -2.34%
Проверка: должно быть 4739 и [35243410, 34419520]:
4739 [35243410. 34419520.]
Для менеджера: после удаления дублей, TEST и отмен выручка упала примерно на 2,3%, как в ДЗ 2.


## [База] 2. Таблица «категория × месяц»
Соберите таблицу 5 × 2 по образцу таблицы городов с урока (категории: `np.unique(cat)`).
Какая категория приносит больше всего выручки за два месяца (`axis`)? Какая упала сильнее всех в %?

In [10]:
# 2. Таблица «категория × месяц»: строки — категории, столбцы — месяцы
categories = np.unique(cat)
category_revenue = np.array([
    [amount[clean & (cat == c) & (month == m)].sum() for m in months]
    for c in categories
])
category_totals = category_revenue.sum(axis=1)  # сумма двух месяцев по строкам
category_changes_pct = (category_revenue[:, 1] / category_revenue[:, 0] - 1) * 100
best_category = categories[np.argmax(category_totals)]
# Выбираем категорию маской: ищем минимальное изменение (самое большое падение)
worst_category = categories[category_changes_pct == category_changes_pct.min()][0]
print("Категория       Сентябрь       Октябрь        Всего    Изменение")
for name, rev, total, change in zip(categories, category_revenue, category_totals, category_changes_pct):
    print(f"{name:<13} {rev[0]:>12,.0f} {rev[1]:>12,.0f} {total:>12,.0f} {change:>10.2f}%")
print("Форма таблицы:", category_revenue.shape)
print(f"Лидер двух месяцев: {best_category}, {category_totals.max():,.0f} ₸")
print(f"Самое сильное падение: {worst_category}, {category_changes_pct.min():.2f}%")
print("Для менеджера: электроника приносит больше всего, а падение в категории «Одежда» требует внимания.")

Категория       Сентябрь       Октябрь        Всего    Изменение
Дом              4,997,730    5,125,790   10,123,520       2.56%
Книги            1,065,300    1,140,910    2,206,210       7.10%
Красота          2,610,950    2,678,990    5,289,940       2.61%
Одежда           9,167,100    8,505,730   17,672,830      -7.21%
Электроника     17,402,330   16,968,100   34,370,430      -2.50%
Форма таблицы: (5, 2)
Лидер двух месяцев: Электроника, 34,370,430 ₸
Самое сильное падение: Одежда, -7.21%
Для менеджера: электроника приносит больше всего, а падение в категории «Одежда» требует внимания.


## [База] 3. Средний чек по категориям в октябре
Соберите вторую таблицу: **число** заказов по категориям и месяцам (сумма маски, а не сумма amount).
Средний чек = выручка / число заказов, одной операцией над двумя таблицами. У какой категории он самый высокий?

In [11]:
# 3. Таблица числа заказов по категориям и месяцам (сумма логической маски)
category_counts = np.array([
    [(clean & (cat == c) & (month == m)).sum() for m in months]
    for c in categories
])
# Делим таблицу выручки на таблицу количества: обе имеют форму (5, 2)
category_avg_check = category_revenue / category_counts
max_oct_index = np.argmax(category_avg_check[:, 1])
print("Категория       Заказов сент.   Заказов окт.    Средний чек окт.")
for name, counts, avgs in zip(categories, category_counts, category_avg_check):
    print(f"{name:<13} {counts[0]:>14} {counts[1]:>14} {avgs[1]:>17,.2f} ₸")
print(f"Наибольший средний чек октября: {categories[max_oct_index]} — {category_avg_check[max_oct_index, 1]:,.2f} ₸")
print("Проверка: всего заказов в таблице и в clean:", category_counts.sum(), clean.sum())
print("Для менеджера: в октябре самый высокий средний чек у электроники — около 33 600 ₸.")

Категория       Заказов сент.   Заказов окт.    Средний чек окт.
Дом                      476            476         10,768.47 ₸
Книги                    265            299          3,815.75 ₸
Красота                  422            435          6,158.60 ₸
Одежда                   678            652         13,045.60 ₸
Электроника              531            505         33,600.20 ₸
Наибольший средний чек октября: Электроника — 33,600.20 ₸
Проверка: всего заказов в таблице и в clean: 4739 4739
Для менеджера: в октябре самый высокий средний чек у электроники — около 33 600 ₸.


## [Стандарт] 4. Доля городов в выручке месяца
Используйте `table` городов с урока. Доля каждого города в выручке своего месяца: `table / table.sum(axis=0)`.
Проверьте, что каждый столбец в сумме даёт 1. Как изменилась доля Шымкента?

In [12]:
# 4. Та же таблица «город × месяц», что и на уроке
cities = np.unique(city)
table = np.array([
    [amount[clean & (city == c) & (month == m)].sum() for m in months]
    for c in cities
])
# axis=0: сумма по городам для каждого месяца; доля города в каждом месяце
city_share = table / table.sum(axis=0)
print("Город          Доля сентября   Доля октября")
for name, s in zip(cities, city_share):
    print(f"{name:<13} {s[0]*100:>12.2f}% {s[1]*100:>14.2f}%")
print("Сумма долей по месяцам:", city_share.sum(axis=0))
print("Проверка (обе суммы равны 1):", np.round(city_share.sum(axis=0), 6) == 1)
# Маска cities == "Шымкент" выбирает только его строку
shymkent_revenue = table[cities == "Шымкент"]
shymkent_share = city_share[cities == "Шымкент"]
print("Выручка Шымкента в двух месяцах:", shymkent_revenue)
print(f"Шымкент: {shymkent_share[0,0]*100:.2f}% -> {shymkent_share[0,1]*100:.2f}%")
print(f"Изменение доли: {(shymkent_share[0,1] - shymkent_share[0,0])*100:.2f} процентного пункта")
print("Для менеджера: доля Шымкента упала; стоит проверить продажи и доставку в этом городе.")

Город          Доля сентября   Доля октября
Актобе                7.31%           7.38%
Алматы               41.15%          42.54%
Астана               25.63%          25.71%
Атырау                2.62%           2.82%
Караганда            10.35%           9.80%
Шымкент              12.93%          11.75%
Сумма долей по месяцам: [1. 1.]
Проверка (обе суммы равны 1): [ True  True]
Выручка Шымкента в двух месяцах: [[4558650. 4044010.]]
Шымкент: 12.93% -> 11.75%
Изменение доли: -1.19 процентного пункта
Для менеджера: доля Шымкента упала; стоит проверить продажи и доставку в этом городе.


## [Стандарт] 5. Кто дал падение
Вклад каждого города в общее падение выручки: изменение города / общее изменение. Сколько процентов падения даёт Шымкент?
Напишите менеджеру одну фразу: где искать причину.

In [13]:
# 5. Изменение выручки каждого города / общее изменение
city_change = table[:, 1] - table[:, 0]
overall_change = city_change.sum()
city_contribution = city_change / overall_change * 100
print(f"Общее изменение выручки: {overall_change:,.0f} ₸")
for name, delta, contribution in zip(cities, city_change, city_contribution):
    print(f"{name:<13} {delta:>12,.0f} ₸    {contribution:>7.2f}%")
# Выбираем Шымкент той же маской cities == "Шымкент"
shymkent_contribution = city_contribution[cities == "Шымкент"][0]
print(f"Вклад Шымкента в падение: {shymkent_contribution:.2f}%")
print("Проверка суммы вкладов (100%):", round(city_contribution.sum(), 6) == 100)
print("Для менеджера: около 62,5% общего падения связано с Шымкентом; проверьте доставку и отмены.")
# Отрицательный вклад: выручка города выросла и уменьшила суммарное падение.

Общее изменение выручки: -823,890 ₸
Актобе             -36,540 ₸       4.44%
Алматы             141,950 ₸     -17.23%
Астана            -183,690 ₸      22.30%
Атырау              45,870 ₸      -5.57%
Караганда         -276,840 ₸      33.60%
Шымкент           -514,640 ₸      62.46%
Вклад Шымкента в падение: 62.46%
Проверка суммы вкладов (100%): True
Для менеджера: около 62,5% общего падения связано с Шымкентом; проверьте доставку и отмены.


## [Стандарт] 6. Исправьте код от AI
AI написал код «доля сентября и октября внутри каждого города». Код падает. Объясните, почему, и исправьте.
```python
share = table / table.sum(axis=1)
```

In [14]:
# 6. Почему ошибается AI: table — (6, 2), а table.sum(axis=1) — (6,)
# Размеры справа (2 и 6) не совпадают, поэтому деление вызывает ValueError.
print("Формы:", table.shape, table.sum(axis=1).shape)
# keepdims=True оставляет суммы строк в форме (6, 1), и деление работает
share_within_city = table / table.sum(axis=1, keepdims=True)
print("Доли сентября и октября внутри каждого города:")
for name, s in zip(cities, share_within_city):
    print(f"{name:<13} {s[0]*100:>6.2f}% {s[1]*100:>6.2f}%")
print("Суммы строк:", share_within_city.sum(axis=1))
print("Проверка: все суммы равны 1:", np.round(share_within_city.sum(axis=1), 6) == 1)
print("Для менеджера: две доли внутри каждого города в сумме дают 100%.")

Формы: (6, 2) (6,)
Доли сентября и октября внутри каждого города:
Актобе         50.36%  49.64%
Алматы         49.76%  50.24%
Астана         50.51%  49.49%
Атырау         48.79%  51.21%
Караганда      51.97%  48.03%
Шымкент        52.99%  47.01%
Суммы строк: [1. 1. 1. 1. 1. 1.]
Проверка: все суммы равны 1: [ True  True  True  True  True  True]
Для менеджера: две доли внутри каждого города в сумме дают 100%.


## [Челлендж] 7. Стандартизация суммы заказа
Для чистых оплаченных заказов посчитайте z-оценку: `z = (x - x.mean()) / x.std()`.
Сколько заказов с |z| > 3 и какая это доля? Проверьте, что у `z` среднее ≈ 0 и std = 1.

In [15]:
# 7. Z-оценка: на сколько стандартных отклонений сумма отличается от среднего
x = amount[clean]
z = (x - x.mean()) / x.std()
outliers = np.abs(z) > 3  # маска заказов, у которых |z| больше 3
outlier_n = outliers.sum()
outlier_share = outlier_n / len(z) * 100
print(f"Чистых оплаченных заказов: {len(z)}")
print(f"Заказов с |z| > 3: {outlier_n}")
print(f"Доля выбросов: {outlier_share:.2f}%")
print("Среднее z:", z.mean(), "| Стандартное отклонение z:", z.std())
print("Проверка: среднее около 0, std около 1:", round(z.mean(), 6) == 0, round(z.std(), 6) == 1)
print("Для менеджера: 87 необычно крупных заказов (около 1,84%) нужно изучить, а не удалять автоматически.")

Чистых оплаченных заказов: 4739
Заказов с |z| > 3: 87
Доля выбросов: 1.84%
Среднее z: 7.04695264417065e-17 | Стандартное отклонение z: 1.0
Проверка: среднее около 0, std около 1: True True
Для менеджера: 87 необычно крупных заказов (около 1,84%) нужно изучить, а не удалять автоматически.


## [Челлендж] 8. Масштабирование признаков (заготовка для ML)
Соберите матрицу `X` из двух столбцов: `amount` и `days` для чистых заказов без пропусков в `days` (`np.column_stack`).
Приведите каждый столбец к диапазону [0, 1] одной строкой: `(X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))`.
Проверьте min и max каждого столбца. Объясните своими словами, зачем модели нужны признаки в одном масштабе.

In [16]:
# 8. Берём чистые заказы с известным сроком доставки
valid = clean & ~np.isnan(days)
# np.column_stack объединяет два столбца в одну таблицу X, как требуется в задании
X = np.column_stack((amount[valid], days[valid]))
# Broadcasting: каждый столбец делится на собственный размах
X_scaled = (X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))
print("Форма X:", X.shape)
print("Минимумы до масштабирования:", X.min(axis=0))
print("Максимумы до масштабирования:", X.max(axis=0))
print("Минимумы после масштабирования:", X_scaled.min(axis=0))
print("Максимумы после масштабирования:", X_scaled.max(axis=0))
print("Первые три строки:")
print(np.round(X_scaled[:3], 4))
print("Проверка минимумов [0, 0]:", np.round(X_scaled.min(axis=0), 6) == 0)
print("Проверка максимумов [1, 1]:", np.round(X_scaled.max(axis=0), 6) == 1)
print("Для менеджера: сумма в тенге и дни доставки теперь в масштабе 0–1; так крупные суммы не перекрывают второй признак.")

Форма X: (4595, 2)
Минимумы до масштабирования: [720.   1.]
Максимумы до масштабирования: [1.3904e+05 8.0000e+00]
Минимумы после масштабирования: [0. 0.]
Максимумы после масштабирования: [1. 1.]
Первые три строки:
[[0.0531 0.    ]
 [0.021  0.1429]
 [0.0607 0.2857]]
Проверка минимумов [0, 0]: [ True  True]
Проверка максимумов [1, 1]: [ True  True]
Для менеджера: сумма в тенге и дни доставки теперь в масштабе 0–1; так крупные суммы не перекрывают второй признак.
